In [3]:
import os
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
project_root = next((
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "core").is_dir()
), None)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root")

os.chdir(project_root)
sys.path.insert(0, str(project_root))
print(project_root)

/home/user/Hehe/allOfMyCode/workspace_1/CoughClassificationProject


In [4]:
import numpy as np

from core.data_pipeline.source_reader import ElderlyCoughAudioSourceReader
from core.data_pipeline.preprocessing.segmentation import CoughSegmenter

In [5]:
source_series = ElderlyCoughAudioSourceReader().get_source_series()
examples = CoughSegmenter(
    kept_metadata_key=["patient_id", "cough_audio", "sample_rate"],
).segment(source_series)

# Original Audio Length

## Time steps

In [6]:
audio_lengths = np.array([len(series.value) for series in source_series])
counts, bins = np.histogram(audio_lengths, bins=20)

for count, start, end in zip(counts, bins[:-1], bins[1:]):
    bar = "█" * round(40 * count / counts.max())
    print(f"{start:>9.0f}–{end:>9.0f} | {count:>4} {bar}")

    52565–    73967 |   20 ████
    73967–    95368 |  120 ████████████████████████
    95368–   116770 |  201 ████████████████████████████████████████
   116770–   138171 |   94 ███████████████████
   138171–   159573 |   34 ███████
   159573–   180975 |   26 █████
   180975–   202376 |   20 ████
   202376–   223778 |   22 ████
   223778–   245179 |   11 ██
   245179–   266581 |    6 █
   266581–   287983 |    9 ██
   287983–   309384 |    3 █
   309384–   330786 |    2 
   330786–   352187 |    3 █
   352187–   373589 |    3 █
   373589–   394991 |    2 
   394991–   416392 |    0 
   416392–   437794 |    1 
   437794–   459195 |    0 
   459195–   480597 |    1 


## Seconds

In [7]:
audio_durations_seconds = np.array([
    len(series.value) / series.metadata["sample_rate"]
    for series in source_series
])
counts, bins = np.histogram(audio_durations_seconds, bins=20)

for count, start, end in zip(counts, bins[:-1], bins[1:]):
    bar = "█" * round(40 * count / counts.max())
    print(f"{start:>7.2f}–{end:>7.2f} s | {count:>4} {bar}")

   3.29–   4.62 s |   20 ████
   4.62–   5.96 s |  120 ████████████████████████
   5.96–   7.30 s |  201 ████████████████████████████████████████
   7.30–   8.64 s |   94 ███████████████████
   8.64–   9.97 s |   34 ███████
   9.97–  11.31 s |   26 █████
  11.31–  12.65 s |   20 ████
  12.65–  13.99 s |   22 ████
  13.99–  15.32 s |   11 ██
  15.32–  16.66 s |    6 █
  16.66–  18.00 s |    9 ██
  18.00–  19.34 s |    3 █
  19.34–  20.67 s |    2 
  20.67–  22.01 s |    3 █
  22.01–  23.35 s |    3 █
  23.35–  24.69 s |    2 
  24.69–  26.02 s |    0 
  26.02–  27.36 s |    1 
  27.36–  28.70 s |    0 
  28.70–  30.04 s |    1 


# Cough Segments

## Time steps

In [8]:
sizes = np.array([example.value.shape[0] for example in examples])
counts, bins = np.histogram(sizes, bins=20)

for count, start, end in zip(counts, bins[:-1], bins[1:]):
    bar = "█" * round(40 * count / counts.max())
    print(f"{start:>7.0f}–{end:>7.0f} | {count:>4} {bar}")

   8193–  14286 |  305 ████████████████████████
  14286–  20379 |  518 ████████████████████████████████████████
  20379–  26471 |  298 ███████████████████████
  26471–  32564 |  154 ████████████
  32564–  38657 |  105 ████████
  38657–  44750 |   40 ███
  44750–  50843 |   19 █
  50843–  56935 |   18 █
  56935–  63028 |   11 █
  63028–  69121 |    5 
  69121–  75214 |    4 
  75214–  81307 |    0 
  81307–  87399 |    5 
  87399–  93492 |    1 
  93492–  99585 |    2 
  99585– 105678 |    2 
 105678– 111771 |    3 
 111771– 117863 |    2 
 117863– 123956 |    2 
 123956– 130049 |    1 


## Seconds

In [9]:
cough_durations_seconds = np.array([
    len(example.value) / example.metadata["sample_rate"]
    for example in examples
])
counts, bins = np.histogram(cough_durations_seconds, bins=20)

for count, start, end in zip(counts, bins[:-1], bins[1:]):
    bar = "█" * round(40 * count / counts.max())
    print(f"{start:>7.2f}–{end:>7.2f} s | {count:>4} {bar}")

   0.51–   0.89 s |  305 ████████████████████████
   0.89–   1.27 s |  518 ████████████████████████████████████████
   1.27–   1.65 s |  298 ███████████████████████
   1.65–   2.04 s |  154 ████████████
   2.04–   2.42 s |  105 ████████
   2.42–   2.80 s |   40 ███
   2.80–   3.18 s |   19 █
   3.18–   3.56 s |   18 █
   3.56–   3.94 s |   11 █
   3.94–   4.32 s |    5 
   4.32–   4.70 s |    4 
   4.70–   5.08 s |    0 
   5.08–   5.46 s |    5 
   5.46–   5.84 s |    1 
   5.84–   6.22 s |    2 
   6.22–   6.60 s |    2 
   6.60–   6.99 s |    3 
   6.99–   7.37 s |    2 
   7.37–   7.75 s |    2 
   7.75–   8.13 s |    1 
